In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [20]:
import sys
from pathlib import Path

# Ensure project root and src are in Python path
PROJECT_ROOT = Path.cwd().parent  # Go from research/ to parent (project root)
SRC_PATH = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
    
print(f"Project root: {PROJECT_ROOT}")
print(f"SRC path: {SRC_PATH}")

# Set environment variables for paths
import os
os.chdir(PROJECT_ROOT)
print(f"Changed working directory to: {os.getcwd()}")


Project root: c:\Users\jayan\Desktop\text-summarization-project
SRC path: c:\Users\jayan\Desktop\text-summarization-project\src
Changed working directory to: c:\Users\jayan\Desktop\text-summarization-project


In [21]:
# Force reload textsummarizer modules to pick up updated constants
import importlib
if 'textsummarizer' in sys.modules:
    del sys.modules['textsummarizer']
if 'textsummarizer.constants' in sys.modules:
    del sys.modules['textsummarizer.constants']
if 'textsummarizer.utils' in sys.modules:
    del sys.modules['textsummarizer.utils']
if 'textsummarizer.logging' in sys.modules:
    del sys.modules['textsummarizer.logging']


In [3]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [29]:
from textsummarizer.constants import PROJECT_ROOT

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        # Convert relative paths to absolute paths
        root_dir = Path(config.root_dir) if Path(config.root_dir).is_absolute() else PROJECT_ROOT / config.root_dir
        local_data_file = Path(config.local_data_file) if Path(config.local_data_file).is_absolute() else PROJECT_ROOT / config.local_data_file
        unzip_dir = Path(config.unzip_dir) if Path(config.unzip_dir).is_absolute() else PROJECT_ROOT / config.unzip_dir

        create_directories([root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=root_dir,
            source_URL=config.source_URL,
            local_data_file=local_data_file,
            unzip_dir=unzip_dir 
        )

        return data_ingestion_config


In [6]:
import os
import urllib.request as request
import zipfile
from textsummarizer.logging import logger
from textsummarizer.utils.common import get_size

In [33]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        # Ensure parent directory exists
        self.config.local_data_file.parent.mkdir(parents=True, exist_ok=True)
        
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")  

        
    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)


In [31]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    if isinstance(e, FileNotFoundError):
        logger.error("Config file not found. Please ensure 'config/config.yaml' exists and the path is correct.")
    else:
        logger.error(f"Error: {type(e).__name__}: {str(e)}")
        import traceback
        traceback.print_exc()


[2026-04-06 10:25:26,411: INFO: common: yaml file: C:\Users\jayan\Desktop\text-summarization-project\config\config.yaml loaded successfully]
[2026-04-06 10:25:26,415: INFO: common: yaml file: C:\Users\jayan\Desktop\text-summarization-project\params.yaml loaded successfully]
[2026-04-06 10:25:26,417: INFO: common: created directory at: artifacts]
[2026-04-06 10:25:26,419: INFO: common: created directory at: C:\Users\jayan\Desktop\text-summarization-project\data_ingestion]
[2026-04-06 10:25:26,670: ERROR: 3211270488: Config file not found. Please ensure 'config/config.yaml' exists and the path is correct.]


In [34]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
    print("✅ Data ingestion completed successfully!")
except Exception as e:
    print(f"❌ Error type: {type(e).__name__}")
    print(f"Error message: {str(e)}")
    import traceback
    traceback.print_exc()


[2026-04-06 10:26:06,276: INFO: common: yaml file: C:\Users\jayan\Desktop\text-summarization-project\config\config.yaml loaded successfully]
[2026-04-06 10:26:06,279: INFO: common: yaml file: C:\Users\jayan\Desktop\text-summarization-project\params.yaml loaded successfully]
[2026-04-06 10:26:06,281: INFO: common: created directory at: artifacts]
[2026-04-06 10:26:06,284: INFO: common: created directory at: C:\Users\jayan\Desktop\text-summarization-project\data_ingestion]
[2026-04-06 10:26:07,827: INFO: 825542103: C:\Users\jayan\Desktop\text-summarization-project\artifacts\data_ingestion\data.zip download! with following info: 
Connection: close
Content-Length: 7903594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "dbc016a060da18070593b83afff580c9b300f0b6ea4147a7988433e04df246ca"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protectio

In [22]:
# Debug: Check the actual paths being used
import sys
from pathlib import Path
print("Current working directory:", Path.cwd())
print("Python path:", sys.path[:3])

from textsummarizer.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
print(f"\nCONFIG_FILE_PATH: {CONFIG_FILE_PATH}")
print(f"PARAMS_FILE_PATH: {PARAMS_FILE_PATH}")
print(f"Config file exists: {CONFIG_FILE_PATH.exists()}")
print(f"Params file exists: {PARAMS_FILE_PATH.exists()}")


Current working directory: c:\Users\jayan\Desktop\text-summarization-project
Python path: ['c:\\Users\\jayan\\Desktop\\text-summarization-project\\src', 'c:\\Users\\jayan\\Desktop\\text-summarization-project', 'C:\\Users\\jayan\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip']

CONFIG_FILE_PATH: C:\Users\jayan\Desktop\text-summarization-project\config\config.yaml
PARAMS_FILE_PATH: C:\Users\jayan\Desktop\text-summarization-project\params.yaml
Config file exists: True
Params file exists: True
